In [9]:
import numpy as np
import struct
from array import array
from os.path  import join
from time import perf_counter

%matplotlib inline
import random
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

In [2]:
# local to this repo

from data import build_dataloaders
from model import ConvModel
from util import test_loss, test_accuracy
from optim import GradSignOptimizer    # my custom optimizer

In [3]:
cross_entropy = nn.CrossEntropyLoss()

In [ ]:
# Just find out how many params
# We will re-initialize the model in-loop for reproducibility

model = ConvModel()
sum([p.numel() for p in model.parameters()])

In [5]:
# Constant across experimental runs
n_batches = 1000


# Stochastic gradient descent

In [ ]:
# training loop

# a good range for learning rate with SGD
for lr in [0.003, 0.01, 0.03, 0.1]:
    # best: Learning rate: 0.03; test accuracy: 0.986


    torch.manual_seed(1)
    model = ConvModel()
    
    train_loader, test_loader = build_dataloaders(seed=1)

    # For reproducibility (see the GradSign section below for why we need this)
    torch.manual_seed(2)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            
            # Uncomment to see partial training progress
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            loss.backward()
    
            with torch.no_grad():
                for p in model.parameters():
                    p -= lr * p.grad
            model.zero_grad()
            
            i += 1
            if i == n_batches:
                break

    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")


# GradSign, my custom optimizer

In [10]:
#training loop
start = perf_counter()
for lr in [0.0001, 0.0003, 0.001, 0.003]:
    # Best: Learning rate: 0.0003; test accuracy: 0.9829
    
    torch.manual_seed(1)
    model = ConvModel()
    train_loader, test_loader = build_dataloaders(seed=1)
    
    optim = GradSignOptimizer(model.named_parameters(), lr=lr)
    model.train()

    # For reproducibility (because optim.__init__() called the torch RNG)
    torch.manual_seed(2)
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)

            # Uncomment to see partial training progress
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            
            i += 1
            if i == n_batches:
                break

    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")
    print(perf_counter() - start)



Computed test accuracy over 10000 items
Learning rate: 0.0001; test accuracy: 0.9765
101.05147150496487
Computed test accuracy over 10000 items
Learning rate: 0.0003; test accuracy: 0.9829
198.5416819229722
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.9829
296.2039322690107
Computed test accuracy over 10000 items
Learning rate: 0.003; test accuracy: 0.977
395.667689393973


In [ ]:
test_accuracy(model, test_loader)

# Adam

In [12]:
#training loop

for lr in [0.0001, 0.0003, 0.001, 0.003]:
    # Best: Learning rate: 0.0003; test accuracy: 0.9849
    
    torch.manual_seed(1)
    model = ConvModel()
    train_loader, test_loader = build_dataloaders(seed=1)
    
    optim = Adam(model.parameters(), lr=lr)
    model.train()
    
    
    i = 0
    while i < n_batches:
        for x, y in train_loader:
            y_pred = model(x)
            loss = cross_entropy(y_pred, y)
            
            # if i % 250 == 0 or i == n_batches - 1:
            #     print(f"Loss at step {i:4} is {loss.item():0.3f}")
            #     print(f"Test loss at step {i:4} is {test_loss(model, test_loader):0.3f}")
        
            optim.zero_grad()
            loss.backward()
            optim.step()
    
            
            i += 1
            if i == n_batches:
                break

    print(f"Learning rate: {lr}; test accuracy: {test_accuracy(model, test_loader)}")



Computed test accuracy over 10000 items
Learning rate: 0.0001; test accuracy: 0.9801
Computed test accuracy over 10000 items
Learning rate: 0.0003; test accuracy: 0.9849
Computed test accuracy over 10000 items
Learning rate: 0.001; test accuracy: 0.984
Computed test accuracy over 10000 items
Learning rate: 0.003; test accuracy: 0.9783
